# [16.8] Do SHAPley and Mechanistic Interpretability Agree? - Solutions

Reference validation notebook for additive agreement, XOR interaction disagreement, and the real CUDA neural-game agreement report.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter16_shapley_attribution_baselines"
section = "part8_shapley_mechinterp_agreement"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part8_shapley_mechinterp_agreement.tests as tests
from chapter16_shapley_attribution_baselines.exercises.part8_shapley_mechinterp_agreement import solutions

GT_TIER = "GT-0"
EXERCISE_ID = "16_8_do_shapley_and_mechanistic_interpretability_agree"
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the CUDA neural-game agreement preflight"
REQUIRES_GPU = True

In [ ]:
tests.test_additive_agreement_smoke_test(solutions.additive_agreement_smoke_test)
tests.test_xor_disagreement_smoke_test(solutions.xor_disagreement_smoke_test)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["additive_agreement"]["agrees_with_mechanistic"]
assert contract["additive_agreement"]["topk_overlap"] == 1.0
assert contract["additive_agreement"]["spearman_correlation"] > 0.99
assert contract["additive_agreement"]["deletion_drop"] > contract["additive_agreement"]["random_baseline_drop"]
assert contract["xor_disagreement"]["ordinary_shapley_misses"]
assert contract["xor_disagreement"]["interaction_recovers_pair"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_family"] == "cuda_trained_neural_coalition_game_mlp"
assert gpu["training_example_count"] == 16
assert gpu["coalition_count"] == 16
assert gpu["fit_mse"] <= 1e-8
assert gpu["spearman_correlation"] > 0.99
assert gpu["topk_overlap"] == 1.0
assert gpu["deletion_drop"] > gpu["random_baseline_drop"]
assert gpu["interaction_max_abs_error"] <= 1e-4
assert gpu["top_interaction_pair"] == [0, 2]
assert gpu["second_interaction_pair"] == [1, 3]
assert gpu["shuffled_control_rejected"]
assert gpu["agreement_artifacts_written"]
assert gpu["agreement_artifact_count"] == 5
assert gpu["agreement_matrix_rows"] >= 7
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "device",
    "spearman_correlation",
    "topk_overlap",
    "deletion_drop",
    "random_baseline_drop",
    "interaction_max_abs_error",
    "peak_vram_gb",
]}